# AzSLD 03 — signer-independent split + benchmark (Phase 0)

**Inputs:** add the outputs of notebook 01 and of all four 02 shards
(*Add Input → Your Work → Notebooks*). **Kaggle settings:** Accelerator **None**; Internet not needed.

AzSLD has no signer ids, so recordings are grouped into pseudo-signers: sessions from the
recording timestamps, then face embeddings (see `azsl/split.py`). One signer → train/val/test
assignment is shared by every task, so a test signer is never seen in training.

**Writes** `splits/`: `signers.csv`, `words200_split.csv`, `glosses_split.csv`,
`sentences_split.csv`, `fingerspelling_split.csv`, `benchmark.json`.
Download `splits/` and commit it to `ml/splits/` — that is the frozen benchmark every model is scored on.

In [ ]:
SIM = 0.60             # sittings at least this alike (average linkage) form one pseudo-signer
LINK_SIM = 0.50        # pseudo-signers with ANY two sittings this alike always share a split
GAP_MIN = 15.0         # minutes between recordings that start a new session
SEED = 0
OVERRIDE_CSV = None    # optional CSV rec_id,signer with real signer ids, if the dataset authors share them

In [ ]:
import glob, json, os, subprocess, sys, tempfile
ON_KAGGLE = os.path.isdir('/kaggle/working')
OUT = '/kaggle/working' if ON_KAGGLE else os.path.abspath('azsl_out')
PKG = os.path.join(tempfile.gettempdir(), 'azsl_pkg')
MODELS = os.path.join(tempfile.gettempdir(), 'azsl_models')
os.makedirs(os.path.join(PKG, 'azsl'), exist_ok=True)
os.makedirs(OUT, exist_ok=True)
if PKG not in sys.path:
    sys.path.insert(0, PKG)

### The `azsl` package
Generated from `ml/azsl/` — edit it there, not here.

In [ ]:
%%writefile {PKG}/azsl/__init__.py
"""AzSLD → MediaPipe landmarks → signer-independent benchmark.

Shared code for the Kaggle notebooks in ml/notebooks/ (they embed this package
verbatim, see ml/build_notebooks.py). Phases 0–1 of the model plan:

  zenodo     read the AzSLD Zenodo archives by byte range (no 44 GB download)
  landmarks  MediaPipe hands + pose + face, same .task models as the browser
  store      landmark arrays packed into part-*.npz files + manifest.csv
  sources    dataset listings: Sentences annotations, Words_200, Fingerspelling
  split      pseudo-signer clustering and the frozen train/val/test split
  pipeline   the steps the notebooks run
"""

In [ ]:
%%writefile {PKG}/azsl/zenodo.py
"""Read AzSLD from Zenodo without downloading whole archives.

Zenodo answers HTTP Range requests, so the central directory of a zip and any
single member can be fetched on their own. AzSLD_Sentences.zip is 43.9 GB — more
than Kaggle keeps — so recordings are read one at a time and only landmarks stay.
"""
import io
import os
import time
import urllib.error
import urllib.request
import zipfile
import zlib
from dataclasses import dataclass

RECORD = '14222948'
ARCHIVES = {
    'words200': 'AzSLD_Words_200.zip',
    'words100': 'AzSLD_Words_100.zip',
    'fingerspelling': 'AzSLD_Fingerspelling.zip',
    'sentences': 'AzSLD_Sentences.zip',
}
UA = {'User-Agent': 'azsl-landmarks/1.0 (+https://github.com/Chevir-App-Development-Team)'}


def archive_url(key: str) -> str:
    return f'https://zenodo.org/records/{RECORD}/files/{ARCHIVES[key]}?download=1'


def http_get(url: str, start: int | None = None, end: int | None = None,
             retries: int = 8, timeout: int = 120) -> bytes:
    """GET (optionally bytes [start, end)) with backoff — Zenodo rate-limits bursts (429)."""
    headers = dict(UA)
    if start is not None:
        headers['Range'] = f'bytes={start}-{end - 1}'
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as r:
                data = r.read()
            if start is not None and len(data) != end - start:
                raise IOError(f'short read: {len(data)} of {end - start} bytes')
            return data
        except urllib.error.HTTPError as e:
            if e.code == 404:
                raise
            last, wait = e, min(300, 5 * 2 ** attempt)
            ra = e.headers.get('Retry-After') if e.headers else None
            if ra and ra.isdigit():
                wait = max(wait, int(ra))
        except (urllib.error.URLError, OSError, TimeoutError) as e:
            last, wait = e, min(300, 5 * 2 ** attempt)
        time.sleep(wait)
    raise IOError(f'GET {url} failed after {retries} tries: {last}')


def download(url: str, dest: str, chunk: int = 1 << 24) -> str:
    """Whole-file download in ranged chunks; resumes if `dest` is partly there."""
    total = int(_head_size(url))
    done = os.path.getsize(dest) if os.path.exists(dest) else 0
    with open(dest, 'ab') as f:
        while done < total:
            end = min(done + chunk, total)
            f.write(http_get(url, done, end))
            done = end
    return dest


def _head_size(url: str, retries: int = 8) -> int:
    req = urllib.request.Request(url, headers={**UA, 'Range': 'bytes=0-0'})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return int(r.headers['Content-Range'].split('/')[-1])
        except (urllib.error.URLError, OSError, TimeoutError) as e:   # DNS blips, 429, resets
            if attempt == retries - 1:
                raise IOError(f'size of {url}: {e}') from e
            time.sleep(min(300, 5 * 2 ** attempt))


class RangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file; just enough for zipfile to read the directory."""

    def __init__(self, url: str, block: int = 1 << 20):
        self.url, self.block, self.pos, self.cache = url, block, 0, {}
        self.size = _head_size(url)

    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = max(0, min(n, self.size - self.pos))
        if not n:
            return b''
        if n > 4 * self.block:                     # big reads (the central directory) in one go
            data = http_get(self.url, self.pos, self.pos + n)
        else:
            out, p = bytearray(), self.pos
            while p < self.pos + n:
                b = p // self.block
                if b not in self.cache:
                    s = b * self.block
                    self.cache[b] = http_get(self.url, s, min(s + self.block, self.size))
                chunk, o = self.cache[b], p - b * self.block
                take = min(len(chunk) - o, self.pos + n - p)
                out += chunk[o:o + take]
                p += take
            data = bytes(out)
        self.pos += len(data)
        return data

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


@dataclass(frozen=True)
class Member:
    """Picklable pointer to one zip member — workers fetch it with a single request."""
    url: str
    name: str
    offset: int          # local header offset
    csize: int
    size: int
    method: int
    crc: int


def list_members(url: str) -> list[Member]:
    z = zipfile.ZipFile(RangeFile(url))
    return [Member(url, i.filename, i.header_offset, i.compress_size, i.file_size, i.compress_type, i.CRC)
            for i in z.infolist() if not i.is_dir()]


def _slack(m: Member) -> int:
    return 30 + len(m.name.encode('utf-8')) + 1024


def _unpack(m: Member, raw: bytes) -> bytes:
    """`raw` starts at the member's local header; returns the verified, decompressed data."""
    if raw[:4] != b'PK\x03\x04':
        raise IOError(f'{m.name}: no local file header at {m.offset}')
    n, e = int.from_bytes(raw[26:28], 'little'), int.from_bytes(raw[28:30], 'little')
    start = 30 + n + e
    data = raw[start:start + m.csize]
    if len(data) < m.csize:
        data = http_get(m.url, m.offset + start, m.offset + start + m.csize)
    if m.method == zipfile.ZIP_DEFLATED:
        data = zlib.decompressobj(-15).decompress(data)
    elif m.method != zipfile.ZIP_STORED:
        raise IOError(f'{m.name}: unsupported compression {m.method}')
    if zlib.crc32(data) != m.crc:
        raise IOError(f'{m.name}: CRC mismatch')
    return data


def read_member(m: Member) -> bytes:
    """Local header + data in one Range request (a second one only if the local extra field is huge)."""
    return _unpack(m, http_get(m.url, m.offset, m.offset + _slack(m) + m.csize))


def read_many(members: list[Member], max_span: int = 8 << 20):
    """Small neighbouring members (annotation JSONs) fetched as shared spans: ~1 request per folder
    instead of one per file. Yields (member, data)."""
    ms = sorted(members, key=lambda m: m.offset)
    i = 0
    while i < len(ms):
        j, end = i, ms[i].offset + _slack(ms[i]) + ms[i].csize
        while j + 1 < len(ms):
            nxt = ms[j + 1]
            nend = nxt.offset + _slack(nxt) + nxt.csize
            if nxt.offset > end + (64 << 10) or nend - ms[i].offset > max_span:
                break
            j, end = j + 1, max(end, nend)
        span = http_get(ms[i].url, ms[i].offset, end)
        for m in ms[i:j + 1]:
            yield m, _unpack(m, span[m.offset - ms[i].offset:])
        i = j + 1

In [ ]:
%%writefile {PKG}/azsl/landmarks.py
"""MediaPipe landmarks per frame, with the same models and settings as the browser.

hand_landmarker.task below is byte-identical to public/tercume/assets/hand_landmarker.task
(same md5), and the hand options match src/tercume/hands.js. Videos run in VIDEO mode
(tracking), like the live camera, so training data has the same jitter and misses
the model will see in the browser. Pose and face use the float16/1 releases that the
browser will load when they are added there.

Per frame:
  hand        (2, 21, 3)  image coords x, y in [0, 1], z relative — slot 0 = signer's RIGHT hand
  hand_world  (2, 21, 3)  metric 3D hand (MediaPipe world landmarks)
  hand_score  (2,)        handedness score of the detection in that slot
  pose        (33, 4)     image coords + visibility
  pose_world  (33, 3)
  face        (102, 3)    lips, eyes, brows, nose, oval, irises (FACE_IDX)
  blend       (52,)       face blendshapes (brows, eyes, mouth — non-manual signals)
Missing detections are NaN. Slots are anatomical: pose wrists decide which hand is which
(same rule as tools/build_words.py), so crossing hands keep their slot.
"""
import base64
import hashlib
import os
import urllib.request

import cv2
import numpy as np

_MP = 'https://storage.googleapis.com/mediapipe-models'
MODELS = {   # name: (url, base64 md5 as GCS reports it)
    'hand': (f'{_MP}/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task', 'FTGEMOo4UWcP6ZFBFqnPrQ=='),
    'pose': (f'{_MP}/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task', 'BKdd33yBGsehpFIyZt19iA=='),
    'face': (f'{_MP}/face_landmarker/face_landmarker/float16/1/face_landmarker.task', 'sOcnSQehZEQE/vZrKN1thQ=='),
}
# OpenCV SFace — only to group recordings by signer for the split; embeddings never leave the split step.
SFACE_URLS = [
    'https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx',
    'https://huggingface.co/opencv/face_recognition_sface/resolve/main/face_recognition_sface_2021dec.onnx',
]
SFACE_FILE = 'face_recognition_sface_2021dec.onnx'
SFACE_MD5 = 'lDaT6px939mmbce36NUqTw=='

FACE_IDX = [
    # lips (outer + inner)
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    # eyes
    263, 249, 390, 373, 374, 380, 381, 382, 362, 466, 388, 387, 386, 385, 384, 398,
    33, 7, 163, 144, 145, 153, 154, 155, 133, 246, 161, 160, 159, 158, 157, 173,
    # eyebrows
    276, 283, 282, 295, 285, 300, 293, 334, 296, 336,
    46, 53, 52, 65, 55, 70, 63, 105, 66, 107,
    # nose, face oval extremes (head pose), iris centres
    1, 2, 98, 327, 10, 152, 234, 454, 468, 473,
]
FIELDS = {
    'hand': (2, 21, 3), 'hand_world': (2, 21, 3), 'hand_score': (2,),
    'pose': (33, 4), 'pose_world': (33, 3),
    'face': (len(FACE_IDX), 3), 'blend': (52,),
}
P_NOSE, P_L_SH, P_R_SH, P_L_WR, P_R_WR = 0, 11, 12, 15, 16
EMB_DIM = 128


def _md5_b64(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return base64.b64encode(h.digest()).decode()


def ensure_models(model_dir: str, sface: bool = False) -> dict:
    """Download the .task files (and SFace) once, verify md5. Returns name → local path."""
    os.makedirs(model_dir, exist_ok=True)
    paths = {}
    for name, (url, md5) in MODELS.items():
        p = os.path.join(model_dir, os.path.basename(url))
        if not os.path.exists(p) or _md5_b64(p) != md5:
            urllib.request.urlretrieve(url, p)
        got = _md5_b64(p)
        if got != md5:
            raise IOError(f'{name}: md5 {got} != {md5} — model release changed, landmarks would not match the browser')
        paths[name] = p
    if sface:
        p = os.path.join(model_dir, SFACE_FILE)
        for url in SFACE_URLS:
            if os.path.exists(p) and _md5_b64(p) == SFACE_MD5:
                break
            try:
                urllib.request.urlretrieve(url, p)
            except OSError as e:
                print(f'SFace download failed from {url}: {e}')
        if not os.path.exists(p) or _md5_b64(p) != SFACE_MD5:
            raise IOError('SFace model missing or md5 mismatch')
        paths['sface'] = p
    return paths


def empty(n: int) -> dict:
    return {k: np.full((n, *s), np.nan, np.float32) for k, s in FIELDS.items()}


def assign_hands(wrists_xy, pose_wrists, aspect):
    """Detected hands → [right slot, left slot] (indices into the detections or None).

    Pose wrists decide, so crossed hands keep their side. Without a pose the signer's
    right hand is on the image's left (the recordings are not mirrored). Same rule as
    tools/build_words.py `_assign`.
    """
    slots = [None, None]
    if not wrists_xy:
        return slots
    if pose_wrists is None:
        for s, k in enumerate(sorted(range(len(wrists_xy)), key=lambda k: wrists_xy[k][0])[:2]):
            slots[s] = k
        return slots
    (lx, ly), (rx, ry) = pose_wrists

    def d(k, px, py):
        return np.hypot((wrists_xy[k][0] - px) * aspect, wrists_xy[k][1] - py)

    if len(wrists_xy) >= 2:
        return [0, 1] if d(0, rx, ry) + d(1, lx, ly) <= d(0, lx, ly) + d(1, rx, ry) else [1, 0]
    slots[0 if d(0, rx, ry) <= d(0, lx, ly) else 1] = 0
    return slots


class Extractor:
    """One per worker process. VIDEO-mode landmarkers are recreated per video (tracking state
    must not leak between clips); IMAGE-mode ones are created once and reused."""

    def __init__(self, paths: dict, face: bool = True, embed: bool = False):
        import mediapipe as mp
        from mediapipe.tasks.python import BaseOptions
        from mediapipe.tasks.python import vision
        self.mp, self.V, self.Base = mp, vision, BaseOptions
        self.paths, self.face_on = paths, face
        self.recognizer = cv2.FaceRecognizerSF.create(paths['sface'], '') if embed else None
        self._image_lms = None

    def _create(self, video: bool):
        V, B = self.V, self.Base
        mode = V.RunningMode.VIDEO if video else V.RunningMode.IMAGE
        hand = V.HandLandmarker.create_from_options(V.HandLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['hand']), running_mode=mode, num_hands=2,
            min_hand_detection_confidence=0.5, min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5))
        pose = V.PoseLandmarker.create_from_options(V.PoseLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['pose']), running_mode=mode, num_poses=1))
        face = V.FaceLandmarker.create_from_options(V.FaceLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['face']), running_mode=mode, num_faces=1,
            output_face_blendshapes=True)) if self.face_on else None
        return hand, pose, face

    @staticmethod
    def _close(lms):
        for lm in lms:
            if lm is not None:
                lm.close()

    def _frame(self, out, t, rgb, lms, ts):
        """Fills row t of `out`; returns the 478 face landmarks (or None) for embedding."""
        hand, pose, face = lms
        im = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        if ts is None:
            hr, pr = hand.detect(im), pose.detect(im)
            fr = face.detect(im) if face else None
        else:
            hr, pr = hand.detect_for_video(im, ts), pose.detect_for_video(im, ts)
            fr = face.detect_for_video(im, ts) if face else None
        H, W = rgb.shape[:2]

        wrists = None
        if pr.pose_landmarks:
            pl = pr.pose_landmarks[0]
            out['pose'][t] = [[p.x, p.y, p.z, p.visibility] for p in pl]
            out['pose_world'][t] = [[p.x, p.y, p.z] for p in pr.pose_world_landmarks[0]]
            wrists = (pl[P_L_WR].x, pl[P_L_WR].y), (pl[P_R_WR].x, pl[P_R_WR].y)

        found = hr.hand_landmarks or []
        slots = assign_hands([(h[0].x, h[0].y) for h in found], wrists, W / H)
        for s, k in enumerate(slots):
            if k is None:
                continue
            out['hand'][t, s] = [[p.x, p.y, p.z] for p in found[k]]
            out['hand_world'][t, s] = [[p.x, p.y, p.z] for p in hr.hand_world_landmarks[k]]
            out['hand_score'][t, s] = hr.handedness[k][0].score

        if fr is not None and fr.face_landmarks:
            fl = fr.face_landmarks[0]
            out['face'][t] = [[fl[i].x, fl[i].y, fl[i].z] for i in FACE_IDX]
            if fr.face_blendshapes:
                bs = [c.score for c in fr.face_blendshapes[0]][:FIELDS['blend'][0]]
                out['blend'][t, :len(bs)] = bs
            return fl
        return None

    def video(self, path: str, stride: int = 1, embed_at=(0.25, 0.5, 0.75)):
        """→ (arrays, meta, face) for a video file; face = {'emb', 'thumb'} or None."""
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_hint = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        rows, picks = [], []
        targets = sorted({int(f * n_hint) for f in embed_at}) if (self.recognizer and n_hint) else []
        lms = self._create(video=True)
        try:
            i = -1
            while True:
                ok, bgr = cap.read()
                if not ok:
                    break
                i += 1
                if i % stride:
                    continue
                row = empty(1)
                fl = self._frame(row, 0, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), lms, int(round(i * 1000.0 / fps)))
                rows.append(row)
                if fl is not None and targets and i >= targets[0]:
                    picks.append((bgr, fl))
                    targets.pop(0)
        finally:
            cap.release()
            self._close(lms)
        arrays = {k: np.concatenate([r[k] for r in rows]) if rows else np.zeros((0, *s), np.float32)
                  for k, s in FIELDS.items()}
        meta = summarize(arrays, W, H)
        meta.update(fps=round(fps / stride, 3), width=W, height=H, n_frames=len(rows))
        return arrays, meta, self._embed(picks)

    def image(self, data: bytes):
        """Single image (fingerspelling) → (arrays with T=1, meta)."""
        bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError('could not decode image')
        if self._image_lms is None:
            self._image_lms = self._create(video=False)
        out = empty(1)
        self._frame(out, 0, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), self._image_lms, None)
        H, W = bgr.shape[:2]
        meta = summarize(out, W, H)
        meta.update(fps=0.0, width=W, height=H, n_frames=1)
        return out, meta

    def _embed(self, picks):
        """Mean SFace embedding over a few frames + one aligned 112×112 crop for eyeballing clusters."""
        if not self.recognizer or not picks:
            return None
        feats, thumb = [], None
        for bgr, fl in picks:
            H, W = bgr.shape[:2]
            pt = lambda i: (fl[i].x * W, fl[i].y * H)
            xs, ys = [p.x * W for p in fl], [p.y * H for p in fl]
            (e1, e2), (m1, m2) = sorted([pt(468), pt(473)]), sorted([pt(61), pt(291)])
            row = np.array([[min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys),
                             *e1, *e2, *pt(1), *m1, *m2, 1.0]], np.float32)
            crop = self.recognizer.alignCrop(bgr, row)
            f = self.recognizer.feature(crop).ravel().astype(np.float32)
            feats.append(f / (np.linalg.norm(f) + 1e-9))
            if thumb is None:
                thumb = cv2.imencode('.jpg', crop, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()
        emb = np.mean(feats, axis=0)
        return {'emb': emb / (np.linalg.norm(emb) + 1e-9), 'thumb': thumb}


def summarize(a: dict, W: int, H: int) -> dict:
    """Detection rates + a frontal-camera check (nose offset from shoulder centre / shoulder width)."""
    T = a['hand'].shape[0]
    if not T:
        return dict(hand_rate=0.0, two_hand_rate=0.0, pose_rate=0.0, face_rate=0.0, frontal=float('nan'))
    hands = ~np.isnan(a['hand'][:, :, 0, 0])
    pose_ok = ~np.isnan(a['pose'][:, 0, 0])
    frontal = float('nan')
    if pose_ok.any():
        p = a['pose'][pose_ok]
        sw = np.abs(p[:, P_L_SH, 0] - p[:, P_R_SH, 0]) * W
        off = np.abs(p[:, P_NOSE, 0] - (p[:, P_L_SH, 0] + p[:, P_R_SH, 0]) / 2) * W
        frontal = float(np.median(off / np.maximum(sw, 1.0)))
    return dict(
        hand_rate=round(float(hands.any(1).mean()), 4),
        two_hand_rate=round(float(hands.all(1).mean()), 4),
        pose_rate=round(float(pose_ok.mean()), 4),
        face_rate=round(float((~np.isnan(a['face'][:, 0, 0])).mean()), 4),
        frontal=round(frontal, 4),
    )

In [ ]:
%%writefile {PKG}/azsl/store.py
"""Landmarks on disk: items packed into part-NNNNN.npz files plus one manifest.csv per subset.

A part holds a few hundred items concatenated along time (float16) with `ids` and
`offsets`, so thousands of clips load as a handful of files. Each manifest row says
which part and offset an item lives in, plus its metadata. Writes are resumable:
items already in the manifest are skipped, and a part is written before its rows are
appended, so an interrupted run never leaves manifest rows without data.
"""
import csv
import functools
import glob
import os

import numpy as np

from .landmarks import EMB_DIM, FIELDS

BASE_COLS = ['id', 'part', 'offset', 'length']


class PartWriter:
    def __init__(self, out_dir: str, items_per_part: int = 400, embeddings: bool = False):
        os.makedirs(out_dir, exist_ok=True)
        self.dir, self.per_part, self.embeddings = out_dir, items_per_part, embeddings
        self.manifest = os.path.join(out_dir, 'manifest.csv')
        nums = [int(os.path.basename(p)[5:10]) for p in glob.glob(os.path.join(out_dir, 'part-*.npz'))]
        self.next_part = max(nums, default=-1) + 1
        self.cols = None
        if os.path.exists(self.manifest):
            with open(self.manifest, newline='', encoding='utf-8') as f:
                self.cols = next(csv.reader(f), None)
        self.buf = []

    def done_ids(self) -> set:
        if not os.path.exists(self.manifest):
            return set()
        with open(self.manifest, newline='', encoding='utf-8') as f:
            return {r['id'] for r in csv.DictReader(f)}

    def add(self, item_id: str, arrays: dict, meta: dict, face: dict | None = None):
        self.buf.append((item_id, arrays, meta, face))
        if len(self.buf) >= self.per_part:
            self.flush()

    def flush(self):
        if not self.buf:
            return
        name = f'part-{self.next_part:05d}.npz'
        lengths = [a['hand'].shape[0] for _, a, _, _ in self.buf]
        offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
        payload = {k: np.concatenate([a[k] for _, a, _, _ in self.buf]).astype(np.float16) for k in FIELDS}
        payload['ids'] = np.array([i for i, _, _, _ in self.buf])
        payload['offsets'] = offsets
        if self.embeddings:
            emb = np.full((len(self.buf), EMB_DIM), np.nan, np.float32)
            thumbs, toff = [], [0]
            for j, (_, _, _, face) in enumerate(self.buf):
                if face is not None:
                    emb[j] = face['emb']
                t = (face or {}).get('thumb') or b''
                thumbs.append(np.frombuffer(t, np.uint8))
                toff.append(toff[-1] + len(t))
            payload['emb'] = emb
            payload['thumb'] = np.concatenate(thumbs) if thumbs else np.zeros(0, np.uint8)
            payload['thumb_off'] = np.array(toff, np.int64)
        path = os.path.join(self.dir, name)
        np.savez_compressed(path + '.tmp.npz', **payload)
        os.replace(path + '.tmp.npz', path)

        rows = [{'id': i, 'part': name, 'offset': int(offsets[j]), 'length': lengths[j], **meta}
                for j, (i, _, meta, _) in enumerate(self.buf)]
        if self.cols is None:
            self.cols = BASE_COLS + [k for k in rows[0] if k not in BASE_COLS]
        extra = set(rows[0]) - set(self.cols)
        if extra:
            raise ValueError(f'new manifest columns {extra} — keep metadata keys stable within a subset')
        new = not os.path.exists(self.manifest)
        with open(self.manifest, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=self.cols)
            if new:
                w.writeheader()
            w.writerows(rows)
        self.buf = []
        self.next_part += 1

    close = flush


class LandmarkStore:
    """Read side. `dirs` may span several outputs (e.g. one per Sentences shard)."""

    def __init__(self, dirs):
        import pandas as pd
        dirs = [dirs] if isinstance(dirs, str) else list(dirs)
        frames = []
        for d in dirs:
            m = os.path.join(d, 'manifest.csv')
            if os.path.exists(m):
                df = pd.read_csv(m, dtype={'id': str}, keep_default_na=False, na_values=[''])
                df['dir'] = d
                frames.append(df)
        if not frames:
            raise FileNotFoundError(f'no manifest.csv in {dirs}')
        self.manifest = pd.concat(frames, ignore_index=True).drop_duplicates('id', keep='last')
        self.index = self.manifest.set_index('id')

    def __len__(self):
        return len(self.manifest)

    @functools.lru_cache(maxsize=16)
    def _part(self, path):
        with np.load(path) as z:
            return {k: z[k] for k in z.files}

    def get(self, item_id: str, dtype=np.float32) -> dict:
        r = self.index.loc[item_id]
        p = self._part(os.path.join(r['dir'], r['part']))
        s, n = int(r['offset']), int(r['length'])
        return {k: p[k][s:s + n].astype(dtype) for k in FIELDS}

    def embeddings(self):
        """→ (ids, emb [N, 128], thumbs list[bytes]) over every part that stored them."""
        ids, embs, thumbs = [], [], []
        for d, part in self.manifest[['dir', 'part']].drop_duplicates().itertuples(index=False):
            p = self._part(os.path.join(d, part))
            if 'emb' not in p:
                continue
            off = p['thumb_off']
            for j, i in enumerate(p['ids']):
                ids.append(str(i))
                embs.append(p['emb'][j])
                thumbs.append(p['thumb'][off[j]:off[j + 1]].tobytes())
        return ids, (np.stack(embs) if embs else np.zeros((0, EMB_DIM), np.float32)), thumbs

In [ ]:
%%writefile {PKG}/azsl/sources.py
"""What is inside the AzSLD archives, as tables.

Layout (checked against Zenodo 14222948, Oct 2026):
  AzSLD_Sentences/<folder>/{Cam1,Cam2}/<YYYY-MM-DD HH-MM-SS>.mp4   + <folder>/ann/<same>.mp4.json
      Cam2 is the FRONTAL camera, and annotations belong to Cam2 (frame counts match);
      Cam1 is the side view and can be a few frames shorter. 1280×960, 30 fps.
      ann JSON (Supervisely): tags[{name, frameRange [start, end] 1-based inclusive, key}]
  AzSLD_Sentences/sentence_list.csv                                folder → sentence, gloss sequence
  AzSLD_Words_200/<GLOSS>/<key>.mp4    clips cut from Cam2 at a tag's frameRange, no margin;
      <key> is the tag key, which links a clip to its recording (and so to its signer)
  AzSLD_Fingerspelling/<LETTER>/*.jpg  40 images per static letter, ~800 frames per dynamic
      letter. "ID--<n>--__<uuid>.jpg" names carry a contributor id; others are numbered frames.
No signer ids anywhere — see split.py for how recordings are grouped by signer.
"""
import csv
import hashlib
import io
import json
import re
from datetime import datetime

from .zenodo import Member, read_many

DYNAMIC_LETTERS = set('çdgköüyz')          # same split as tools/build_poses.py
RUN_GAP = 20                               # frame-number gap that starts a new clip


def az_lower(s: str) -> str:
    """Azerbaijani lower-case: I → ı, İ → i (str.lower() gets these wrong)."""
    return s.replace('I', 'ı').replace('İ', 'i').lower()


def anon(raw: str) -> str:
    """Contributor ids in file names are Telegram user ids — keep only a short hash."""
    return hashlib.sha1(raw.encode()).hexdigest()[:10]


def parse_stamp(stem: str):
    m = re.match(r'(\d{4}-\d\d-\d\d \d\d-\d\d-\d\d)', stem)
    return datetime.strptime(m.group(1), '%Y-%m-%d %H-%M-%S') if m else None


def sentence_recordings(members: list[Member], camera: str = 'Cam2') -> list[Member]:
    """Video members of one camera, in a stable order."""
    rx = re.compile(rf'^AzSLD_Sentences/([^/]+)/{camera}/([^/]+)\.mp4$')
    return sorted((m for m in members if rx.match(m.name)), key=lambda m: m.name)


def rec_id(name: str) -> str:
    """'AzSLD_Sentences/12/Cam2/2022-06-01 15-17-31.mp4' → '12/2022-06-01 15-17-31'."""
    p = name.split('/')
    stem = p[-1]
    for suf in ('.mp4.json', '.mp4'):
        if stem.endswith(suf):
            stem = stem[:-len(suf)]
    return f'{p[1]}/{stem}'


def sentences_index(members: list[Member]):
    """→ (tags, recordings, sentences) DataFrames from the annotation JSONs + sentence_list.csv."""
    import pandas as pd
    anns = [m for m in members if re.match(r'^AzSLD_Sentences/[^/]+/ann/.+\.json$', m.name)]
    lst = [m for m in members if m.name.endswith('sentence_list.csv')]
    cams = {c: {rec_id(m.name): m.size for m in sentence_recordings(members, c)} for c in ('Cam1', 'Cam2')}

    tags, recs = [], []
    for m, data in read_many(anns):
        rid = rec_id(m.name)
        d = json.loads(data.decode('utf-8'))
        recs.append({
            'rec_id': rid, 'folder': rid.split('/')[0], 'stamp': parse_stamp(rid.split('/')[1]),
            'frames': d.get('framesCount'), 'width': d.get('size', {}).get('width'),
            'height': d.get('size', {}).get('height'), 'n_tags': len(d.get('tags', [])),
            'cam1': rid in cams['Cam1'], 'cam2': rid in cams['Cam2'],
        })
        for t in d.get('tags', []):
            fr = t.get('frameRange') or [None, None]
            tags.append({'key': t['key'], 'gloss': t['name'], 'rec_id': rid, 'start': fr[0], 'end': fr[1],
                         'labeler': t.get('labelerLogin'), 'created': t.get('createdAt')})
    tags, recs = pd.DataFrame(tags), pd.DataFrame(recs)
    tags['dup'] = _duplicates(tags)

    sentences = pd.DataFrame(columns=['folder', 'text', 'glosses'])
    if lst:
        rows = list(csv.reader(io.StringIO(next(read_many(lst))[1].decode('utf-8-sig'))))
        sentences = pd.DataFrame([{'folder': r[0].strip(), 'text': r[1].strip(), 'glosses': r[2].strip()}
                                  for r in rows[1:] if len(r) >= 3])
    return tags, recs, sentences


def _duplicates(tags):
    """Same gloss annotated twice over the same span of one recording (seen in the data) → later one is dup."""
    dup = [False] * len(tags)
    if tags.empty:
        return dup
    for _, g in tags.sort_values('created').groupby(['rec_id', 'gloss']):
        kept = []
        for idx, r in g.iterrows():
            s, e = r['start'], r['end']
            if any(min(e, ke) - max(s, ks) + 1 >= 0.5 * min(e - s + 1, ke - ks + 1) for ks, ke in kept):
                dup[tags.index.get_loc(idx)] = True
            else:
                kept.append((s, e))
    return dup


def words_listing(names: list[str], root: str = 'AzSLD_Words_200'):
    """Zip member names → key (tag key, links to a recording), gloss, member."""
    import pandas as pd
    rx = re.compile(rf'^{root}/([^/]+)/([0-9a-f]+)\.mp4$')
    rows = [{'key': mm.group(2), 'gloss': mm.group(1), 'member': n} for n in names if (mm := rx.match(n))]
    return pd.DataFrame(rows)


def fingerspelling_listing(names: list[str]):
    """One row per image: letter, contributor group (hashed) or clip run, frame number."""
    import pandas as pd
    rows = []
    for name in names:
        p = name.split('/')
        if len(p) != 3 or not p[2].lower().endswith('.jpg'):
            continue
        letter, file = az_lower(p[1]), p[2]
        cid = re.match(r'^ID--(\d+)--', file)
        num = re.search(r'(\d+)\.jpg$', file, re.I)
        rows.append({'file': file, 'member': name, 'letter': letter,
                     'kind': 'dynamic' if letter in DYNAMIC_LETTERS else 'static',
                     'contributor': anon(cid.group(1)) if cid else '',
                     'frame_no': int(num.group(1)) if (num and not cid) else -1})
    df = pd.DataFrame(rows)
    # numbered frames: a gap > RUN_GAP starts a new clip (as in tools/build_poses.py)
    df['run'] = ''
    for letter, g in df[df.contributor == ''].sort_values('frame_no').groupby('letter'):
        run, prev = 0, None
        for idx, n in g['frame_no'].items():
            if prev is not None and n - prev > RUN_GAP:
                run += 1
            df.at[idx, 'run'] = f'{letter}-run{run}'
            prev = n
    df['group'] = df.contributor.where(df.contributor != '', df.run)
    return df

In [ ]:
%%writefile {PKG}/azsl/split.py
"""Signer-independent splits for AzSLD.

AzSLD ships no signer ids and Words_200 clips are UUID-named, which is how the old
I3D repo ended up with the same signers in train and test. Here every Sentences
recording gets a pseudo-signer:

  1. sessions — recordings ordered by their file-name timestamp; a gap of more than
     `gap_min` minutes (or a new day) starts a new session. One sitting ≈ one signer.
  2. faces — a mean SFace embedding per recording (landmarks.Extractor._embed).
     Recordings far from their session's centroid are split off (a second person
     in that sitting), then session cores + split-offs are clustered (average
     linkage, cosine) at similarity `sim`.
  3. one signer → split assignment, chosen so the 200-word vocabulary is covered in
     val/test, and reused for every task (words, sentence-cut glosses, sentences),
     so a test signer is never seen in training by any model.

If the dataset authors can provide real signer ids, pass them as an override CSV
(rec_id, signer) and steps 1–2 are skipped for those recordings.
"""
import numpy as np


def sessions(recs, gap_min: float = 15.0):
    """recs: DataFrame with rec_id, stamp → Series rec_id → session id."""
    import pandas as pd
    r = recs.dropna(subset=['stamp']).sort_values('stamp')
    sid, prev, out = -1, None, {}
    for rid, t in zip(r.rec_id, r.stamp):
        if prev is None or t.date() != prev.date() or (t - prev).total_seconds() > gap_min * 60:
            sid += 1
        out[rid] = f's{sid:04d}'
        prev = t
    return pd.Series(out, name='session')


def _norm(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


def cluster_signers(rec_ids, emb, session_of: dict, sim: float = 0.60):
    """→ (dict rec_id → signer, report dict). `emb` rows are L2-normalised SFace means."""
    from sklearn.cluster import AgglomerativeClustering
    by_session = {}
    for i, rid in enumerate(rec_ids):
        by_session.setdefault(session_of.get(rid, f'nosession-{rid}'), []).append(i)

    units, members = [], []                      # unit embedding, indices of its recordings
    split_off = 0
    for idx in by_session.values():
        core = list(idx)
        for _ in range(2):                       # centroid, drop far members, centroid again
            c = _norm(emb[core].mean(0))
            keep = [i for i in core if emb[i] @ c >= sim]
            if not keep or len(keep) == len(core):
                break
            core = keep
        units.append(_norm(emb[core].mean(0)))
        members.append(core)
        for i in idx:
            if i not in core:
                units.append(emb[i])
                members.append([i])
                split_off += 1
    U = np.stack(units)
    if len(U) == 1:
        labels = np.zeros(1, int)
    else:
        labels = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                         distance_threshold=1 - sim).fit_predict(U)
    signer = {}
    for lab, mem in zip(labels, members):
        for i in mem:
            signer[rec_ids[i]] = f'p{lab:03d}'
    # similarity of each unit to its nearest other cluster — a clean gap means the threshold is safe
    cents = {lab: _norm(U[labels == lab].mean(0)) for lab in set(labels)}
    C = np.stack(list(cents.values()))
    S = C @ C.T
    np.fill_diagonal(S, -1)
    report = {'units': len(U), 'split_off_recordings': split_off, 'signers': len(cents),
              'nearest_other_signer_sim': [round(float(x), 3) for x in np.sort(S.max(1))[::-1][:10]]}
    return signer, report


def assign_signers(recs, emb_ids, emb, sim=0.60, gap_min=15.0, override=None):
    """recs → DataFrame rec_id, session, signer, has_emb. Recordings without a face fall back to
    their session's signer; with neither they become 'unk-<session>'."""
    ses = sessions(recs, gap_min)
    out = recs[['rec_id']].copy()
    out['session'] = out.rec_id.map(ses).fillna('nosession')
    has = set(emb_ids)
    out['has_emb'] = out.rec_id.isin(has)
    signer, report = ({}, {'signers': 0})
    if len(emb_ids):
        signer, report = cluster_signers(list(emb_ids), _norm(np.asarray(emb, np.float32)), ses.to_dict(), sim)
    out['signer'] = out.rec_id.map(signer)
    maj = out.dropna(subset=['signer']).groupby('session').signer.agg(lambda s: s.value_counts().index[0])
    miss = out.signer.isna()
    out.loc[miss, 'signer'] = out.loc[miss, 'session'].map(maj)
    out['signer'] = out.signer.fillna('unk-' + out.session)
    if override is not None and len(override):
        o = dict(zip(override.rec_id, override.signer.astype(str)))
        out['signer'] = [o.get(r, s) for r, s in zip(out.rec_id, out.signer)]
        report['override'] = len(o)
    # same signer recording the same sentence twice is rare (retakes); many means merged people
    folder = out.rec_id.str.split('/').str[0]
    rep = (out.assign(folder=folder).groupby(['signer', 'folder']).size() > 1).sum()
    report['signer_sentence_repeats'] = int(rep)
    report['recordings'] = len(out)
    report['without_face'] = int((~out.has_emb).sum())
    return out, report


def _units(signers, emb_ids, emb):
    """(session, signer) units — one sitting of one person — and their centroids."""
    sig = dict(zip(signers.rec_id, signers.signer))
    ses = dict(zip(signers.rec_id, signers.session))
    E = np.asarray(emb, np.float32)
    groups = {}
    for i, rid in enumerate(emb_ids):
        if rid in sig:
            groups.setdefault((ses[rid], sig[rid]), []).append(i)
    keys = sorted(groups)
    C = np.stack([_norm(E[groups[k]].mean(0)) for k in keys]) if keys else np.zeros((0, 1))
    return keys, C, {k: len(v) for k, v in groups.items()}


def split_groups(signers, emb_ids, emb, link_sim=0.50):
    """Tie pseudo-signers that share a look-alike sitting: if ANY sitting of one is >= link_sim
    similar to any sitting of the other, they always land in the same split. Comparing sittings,
    not cluster averages, matters — a cluster that mixes two people has an average resembling
    neither. On AzSLD one person's sittings score 0.6+ and different people mostly < 0.5.
    Union-find over the pairs → dict signer → group id."""
    keys, C, _ = _units(signers, emb_ids, emb)
    parent = {n: n for n in signers.signer.unique()}

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    if len(keys) > 1:
        I, J = np.where(np.triu(C @ C.T, 1) >= link_sim)
        for i, j in zip(I, J):
            if keys[i][1] != keys[j][1]:
                parent[root(keys[i][1])] = root(keys[j][1])
    roots = sorted({root(n) for n in parent})
    gid = {r: f'g{k:02d}' for k, r in enumerate(roots)}
    return {n: gid[root(n)] for n in parent}


def closest_pairs(signers, emb_ids, emb, top=8, cross_split_only=False):
    """Most similar sittings of different pseudo-signers, with their splits — read next to the
    contact sheet. With split_groups() no pair at or above link_sim straddles two splits."""
    keys, C, n = _units(signers, emb_ids, emb)
    split = dict(zip(signers.signer, signers.split))
    S = np.triu(C @ C.T, 1)
    order = np.dstack(np.unravel_index(np.argsort(-S, axis=None), S.shape))[0]
    out = []
    for i, j in order:
        if j <= i or keys[i][1] == keys[j][1]:
            continue
        a, b = keys[i], keys[j]
        if cross_split_only and split.get(a[1]) == split.get(b[1]):
            continue
        out.append({'a': f'{a[1]} {a[0]}', 'b': f'{b[1]} {b[0]}', 'sim': round(float(S[i, j]), 3),
                    'splits': f'{split.get(a[1])}/{split.get(b[1])}', 'recordings': f'{n[a]}/{n[b]}'})
        if len(out) >= top:
            break
    return out


def pick_split(df, group_col: str, label_col: str, fracs=(0.7, 0.15, 0.15), seed: int = 0,
               trials: int = 4000, forced_train=()):
    """Random search over whole groups → dict group → 'train'|'val'|'test'.

    Score: classes (with ≥3 groups) missing from test/val, then distance from target
    sample fractions. Groups in `forced_train` (unknown provenance) never go to val/test.
    """
    rng = np.random.default_rng(seed)
    counts = df.groupby(group_col).size()
    labels_of = df.groupby(group_col)[label_col].agg(set)
    groups = [g for g in counts.index if g not in set(forced_train)]
    n_groups_per_label = df[~df[group_col].isin(forced_train)].groupby(label_col)[group_col].nunique()
    testable = set(n_groups_per_label[n_groups_per_label >= 3].index)
    N = counts[groups].sum()
    best, best_score = None, None
    for _ in range(trials):
        order = rng.permutation(groups)
        assign, n_test, n_val = {}, 0, 0
        for g in order:
            if n_test < fracs[2] * N:
                assign[g], n_test = 'test', n_test + counts[g]
            elif n_val < fracs[1] * N:
                assign[g], n_val = 'val', n_val + counts[g]
            else:
                assign[g] = 'train'
        cov = {s: set().union(*[labels_of[g] for g in assign if assign[g] == s] or [set()]) for s in ('val', 'test')}
        miss_t, miss_v = len(testable - cov['test']), len(testable - cov['val'])
        dev = abs(n_test / N - fracs[2]) + abs(n_val / N - fracs[1])
        score = (miss_t, miss_v, round(float(dev), 3))
        if best_score is None or score < best_score:
            best, best_score = assign, score
    for g in forced_train:
        best[g] = 'train'
    return best, {'missing_test': best_score[0], 'missing_val': best_score[1],
                  'fraction_error': best_score[2], 'testable_classes': len(testable)}


def folds(df, group_col: str, k: int = 5):
    """Train groups → fold 0..k-1, balancing sample counts (largest groups first)."""
    sizes = df.groupby(group_col).size().sort_values(ascending=False)
    load, out = [0] * k, {}
    for g, n in sizes.items():
        f = int(np.argmin(load))
        out[g], load[f] = f, load[f] + n
    return out


def baselines(df, label_col: str, split_col: str = 'split') -> dict:
    """Per split: samples, classes, and what always answering the most frequent train class scores."""
    train = df[df[split_col] == 'train']
    top = train[label_col].value_counts()
    major = top.index[0] if len(top) else None
    out = {'majority_class': major, 'train_classes': int(train[label_col].nunique())}
    for s in ('train', 'val', 'test'):
        d = df[df[split_col] == s]
        out[s] = {
            'samples': int(len(d)),
            'classes': int(d[label_col].nunique()),
            'majority_acc': round(float((d[label_col] == major).mean()), 4) if len(d) else None,
            'chance_balanced_acc': round(1 / max(1, d[label_col].nunique()), 4),
            'unseen_in_train': int(len(set(d[label_col]) - set(train[label_col]))) if s != 'train' else 0,
        }
    return out

In [ ]:
%%writefile {PKG}/azsl/pipeline.py
"""The steps the Kaggle notebooks run. Every step is resumable and takes `limit` for smoke tests
(with `limit`, archives are read member-by-member over HTTP instead of being downloaded)."""
import json
import multiprocessing as mp
import os
import shutil
import tempfile
import time
import zipfile
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from concurrent.futures.process import BrokenProcessPool

from . import landmarks, sources, store, zenodo
from . import split as S

_EX = None
_ZIP = None


# ---------------------------------------------------------------- workers

def _init_worker(paths, face, embed, zip_path):
    global _EX, _ZIP
    # MediaPipe's C++ side logs ~20 warnings per landmarker; thousands of videos would flood the Kaggle log
    os.environ.setdefault('GLOG_minloglevel', '2')
    os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
    import cv2
    cv2.setNumThreads(1)                     # one process per core already
    _EX = landmarks.Extractor(paths, face=face, embed=embed)
    _ZIP = zipfile.ZipFile(zip_path) if zip_path else None


def _read(src):
    return _ZIP.read(src) if isinstance(src, str) else zenodo.read_member(src)


def _video_job(task):
    item_id, src, extra = task
    t0 = time.time()
    data = _read(src)
    fd, tmp = tempfile.mkstemp(suffix='.mp4')
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(data)
        arrays, meta, face = _EX.video(tmp)
    finally:
        os.remove(tmp)
    meta.update(extra)
    meta['sec'] = round(time.time() - t0, 2)
    return item_id, arrays, meta, face


def _image_job(task):
    item_id, src, extra = task
    arrays, meta = _EX.image(_read(src))
    meta.update(extra)
    return item_id, arrays, meta, None


def run_pool(tasks, job, writer, init_args, workers=None, time_budget_h=11.0, log_every=250, label=''):
    """Runs `job` over tasks [(id, source, extra meta)] in spawned processes, writing as results come.
    Stops taking new tasks after `time_budget_h` so a Kaggle run can finish and save its output.
    A crashed worker (MediaPipe segfault) restarts the pool; its in-flight tasks get one retry."""
    workers = workers or os.cpu_count() or 2
    done = writer.done_ids()
    queue = deque(t for t in tasks if t[0] not in done)
    total = len(queue)
    workers = max(1, min(workers, total))
    print(f'{label}: {len(tasks)} items, {len(done)} already done, {total} to go, {workers} workers', flush=True)
    t0, deadline = time.time(), time.time() + time_budget_h * 3600
    errors, retries, n, frames = [], {}, 0, 0
    ctx = mp.get_context('spawn')
    fruitless = 0                            # pool restarts in a row without a finished item
    while queue and time.time() < deadline:
        pending, n_start = {}, n
        try:
            with ProcessPoolExecutor(workers, mp_context=ctx, initializer=_init_worker, initargs=init_args) as ex:
                while queue or pending:
                    while queue and len(pending) < workers * 3 and time.time() < deadline:
                        t = queue.popleft()
                        try:
                            pending[ex.submit(job, t)] = t
                        except BrokenProcessPool:
                            queue.appendleft(t)      # not in `pending` yet — would be lost
                            raise
                    if not pending:
                        break
                    finished, _ = wait(pending, return_when=FIRST_COMPLETED)
                    for f in finished:
                        t = pending.pop(f)
                        try:
                            item_id, arrays, meta, face = f.result()
                        except BrokenProcessPool:
                            raise
                        except Exception as e:  # noqa: BLE001 — one bad file must not stop the run
                            errors.append({'id': t[0], 'error': repr(e)[:300]})
                            continue
                        writer.add(item_id, arrays, meta, face)
                        n, frames = n + 1, frames + meta['n_frames']
                        if n % log_every == 0:
                            el = time.time() - t0
                            eta = el / n * (total - n) / 3600
                            print(f'  {n}/{total}  {frames / el:.0f} frames/s  {el / 3600:.2f} h  ETA {eta:.2f} h',
                                  flush=True)
        except BrokenProcessPool:
            fruitless = fruitless + 1 if n == n_start else 0
            if fruitless >= 3:
                writer.flush()
                raise RuntimeError('workers keep crashing before finishing anything — mediapipe or a model '
                                   'is broken in this environment (run the self-test cell)')
            print('  worker crashed — restarting pool', flush=True)
            for t in pending.values():
                retries[t[0]] = retries.get(t[0], 0) + 1
                if retries[t[0]] <= 1:
                    queue.appendleft(t)
                else:
                    errors.append({'id': t[0], 'error': 'worker crashed twice'})
    writer.flush()
    if errors:
        import csv
        path = os.path.join(writer.dir, 'errors.csv')
        new = not os.path.exists(path)
        with open(path, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['id', 'error'])
            if new:
                w.writeheader()
            w.writerows(errors)
    el = time.time() - t0
    left = total - n - len(errors)
    summary = {'done': n, 'errors': len(errors), 'remaining': left, 'hours': round(el / 3600, 2),
               'frames_per_s': round(frames / el, 1) if el else 0.0, 'complete': left == 0}
    print(f'{label}: {summary}', flush=True)
    if left:
        print(f'  {left} items left (time budget) — rerun with this output attached to resume', flush=True)
    return summary


# ---------------------------------------------------------------- steps

def setup_models(model_dir='/tmp/azsl_models', sface=False):
    return landmarks.ensure_models(model_dir, sface=sface)


def mediapipe_ok(paths):
    """Creates a hand landmarker in a fresh process (a broken mediapipe/numpy pairing can crash
    the interpreter). → mediapipe version, or None."""
    import subprocess
    import sys
    code = ('import mediapipe as mp, numpy as np\n'
            'from mediapipe.tasks.python import vision as V, BaseOptions as B\n'
            f'h = V.HandLandmarker.create_from_options(V.HandLandmarkerOptions(base_options=B(model_asset_path={paths["hand"]!r})))\n'
            'h.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.zeros((64, 64, 3), np.uint8)))\n'
            'print(mp.__version__)')
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:])
        return None
    return r.stdout.strip().splitlines()[-1]


def run_index(out_dir, folders=None):
    """Sentences annotations → index/tags.csv, recordings.csv, sentences.csv (a few MB, ~minutes)."""
    members = zenodo.list_members(zenodo.archive_url('sentences'))
    if folders is not None:
        keep = {str(f) for f in folders}
        members = [m for m in members if m.name.split('/')[1] in keep or m.name.endswith('sentence_list.csv')]
    tags, recs, sents = sources.sentences_index(members)
    d = os.path.join(out_dir, 'index')
    os.makedirs(d, exist_ok=True)
    tags.to_csv(os.path.join(d, 'tags.csv'), index=False)
    recs.to_csv(os.path.join(d, 'recordings.csv'), index=False)
    sents.to_csv(os.path.join(d, 'sentences.csv'), index=False)
    print(f'index: {len(recs)} recordings ({int(recs.cam2.sum())} with Cam2), {len(tags)} tags '
          f'({int(tags.dup.sum())} duplicates), {tags.gloss.nunique()} glosses, {len(sents)} sentences')
    return tags, recs, sents


def _archive(key, tmp_dir, limit):
    """Full runs download the archive (one big request is kinder to Zenodo than thousands);
    smoke tests read the needed members over HTTP."""
    url = zenodo.archive_url(key)
    if limit:
        return None, zenodo.list_members(url)
    path = os.path.join(tmp_dir, zenodo.ARCHIVES[key])
    t0 = time.time()
    zenodo.download(url, path)
    print(f'{zenodo.ARCHIVES[key]}: {os.path.getsize(path) / 1e9:.2f} GB in {time.time() - t0:.0f} s', flush=True)
    with zipfile.ZipFile(path) as z:
        names = [i.filename for i in z.infolist() if not i.is_dir()]
    return path, names


def _by_name(members):
    """name → task source: the name itself (read from the local zip) or a zenodo.Member (HTTP)."""
    return {m if isinstance(m, str) else m.name: m for m in members}


def run_words200(out_dir, paths, workers=None, limit=None, tmp_dir='/tmp', time_budget_h=11.0):
    zpath, members = _archive('words200', tmp_dir, limit)
    by_name = _by_name(members)
    listing = sources.words_listing(list(by_name))
    if limit:
        listing = listing.groupby('gloss').head(1).head(limit)
    tasks = [(r.key, by_name[r.member], {'key': r.key, 'gloss': r.gloss}) for r in listing.itertuples()]
    writer = store.PartWriter(os.path.join(out_dir, 'landmarks', 'words200'), items_per_part=1000)
    return run_pool(tasks, _video_job, writer, (paths, True, False, zpath), workers, time_budget_h,
                    label='Words_200')


def run_fingerspelling(out_dir, paths, workers=None, limit=None, tmp_dir='/tmp', time_budget_h=11.0):
    zpath, members = _archive('fingerspelling', tmp_dir, limit)
    by_name = _by_name(members)
    listing = sources.fingerspelling_listing(list(by_name))
    if limit:
        listing = listing.groupby('letter').head(1).head(limit)
    tasks = [(r.member.split('/', 1)[1], by_name[r.member],
              {'letter': r.letter, 'kind': r.kind, 'group': r.group, 'run': r.run, 'frame_no': r.frame_no})
             for r in listing.itertuples()]
    writer = store.PartWriter(os.path.join(out_dir, 'landmarks', 'fingerspelling'), items_per_part=4000)
    return run_pool(tasks, _image_job, writer, (paths, True, False, zpath), workers, time_budget_h,
                    log_every=1000, label='Fingerspelling')


def run_sentences(out_dir, paths, shard=0, num_shards=1, camera='Cam2', workers=None, limit=None,
                  time_budget_h=11.0, resume_from=()):
    """One shard of the Sentences recordings, read one by one from the 43.9 GB zip.
    Cam2 (frontal) also stores a face embedding per recording for the signer split."""
    members = zenodo.list_members(zenodo.archive_url('sentences'))
    recs = sources.sentence_recordings(members, camera)
    mine = [m for k, m in enumerate(recs) if k % num_shards == shard]
    if limit:
        mine = mine[:limit]
    sub = os.path.join(out_dir, 'landmarks', f'sentences_{camera.lower()}', f'shard{shard:02d}')
    for prev in resume_from:                 # earlier run of this shard attached as input
        if os.path.exists(os.path.join(prev, 'manifest.csv')) and not os.path.exists(os.path.join(sub, 'manifest.csv')):
            shutil.copytree(prev, sub, dirs_exist_ok=True)
            print(f'resumed from {prev}', flush=True)
    embed = camera == 'Cam2'
    writer = store.PartWriter(sub, items_per_part=200, embeddings=embed)
    tasks = [(sources.rec_id(m.name), m, {'rec_id': sources.rec_id(m.name), 'camera': camera, 'bytes': m.size})
             for m in mine]
    return run_pool(tasks, _video_job, writer, (paths, True, embed, None), workers, time_budget_h,
                    log_every=100, label=f'Sentences {camera} shard {shard}/{num_shards}')


# ---------------------------------------------------------------- split

def run_split(index_dir, words_dirs, fs_dirs, sentence_dirs, out_dir, sim=0.60, link_sim=0.50, gap_min=15.0,
              seed=0, override_csv=None, heldout_sentence_frac=0.1, min_clips=20, k_folds=5, trials=4000):
    """Pseudo-signers → one signer-level train/val/test split → per-task split files + benchmark.json."""
    import numpy as np
    import pandas as pd
    tags = pd.read_csv(os.path.join(index_dir, 'tags.csv'), dtype={'key': str, 'rec_id': str})
    recs = pd.read_csv(os.path.join(index_dir, 'recordings.csv'), dtype={'rec_id': str, 'folder': str},
                       parse_dates=['stamp'])
    sents = pd.read_csv(os.path.join(index_dir, 'sentences.csv'), dtype={'folder': str})

    sen = store.LandmarkStore(sentence_dirs)
    emb_ids, emb, thumbs = sen.embeddings()
    ok = ~np.isnan(emb).any(1) if len(emb) else np.zeros(0, bool)
    emb_ids, emb = [i for i, k in zip(emb_ids, ok) if k], emb[ok]
    # compact copy for tuning the clustering offline (anonymous vectors, no images; never committed)
    priv = os.path.join(os.path.dirname(os.path.abspath(out_dir)), 'private')
    os.makedirs(priv, exist_ok=True)
    np.savez_compressed(os.path.join(priv, 'face_embeddings.npz'), ids=np.array(emb_ids), emb=emb.astype(np.float16))
    override = pd.read_csv(override_csv, dtype=str) if override_csv else None
    signers, rep = S.assign_signers(recs, emb_ids, emb, sim=sim, gap_min=gap_min, override=override)
    group_of = S.split_groups(signers, emb_ids, emb, link_sim)
    signers['split_group'] = signers.signer.map(group_of)
    m = sen.manifest.set_index('rec_id') if 'rec_id' in sen.manifest else sen.manifest.set_index('id')
    signers['has_landmarks'] = signers.rec_id.isin(m.index)
    signers['frontal'] = signers.rec_id.map(m['frontal'])
    signer_of = dict(zip(signers.rec_id, signers.signer))

    words = store.LandmarkStore(words_dirs).manifest[['id', 'gloss', 'hand_rate', 'n_frames']].rename(columns={'id': 'key'})
    vocab = set(words.gloss)
    t = tags[~tags.dup].copy()
    t['signer'] = t.rec_id.map(signer_of)
    t = t.dropna(subset=['signer'])
    t['split_group'] = t.signer.map(group_of)

    # one split over split groups (pseudo-signers + the ones ambiguously alike), optimised for
    # covering the 200-word vocabulary in val and test, then shared by every task
    t200 = t[t.gloss.isin(vocab)]
    unknown = sorted({group_of[g] for g in t.signer.unique() if g.startswith('unk-')})
    assign, cover = S.pick_split(t200, 'split_group', 'gloss', seed=seed, trials=trials, forced_train=unknown)
    for g in set(group_of.values()):
        assign.setdefault(g, 'train')
    fold_of = S.folds(t200[t200.split_group.map(assign) == 'train'], 'split_group', k_folds)
    signers['split'] = signers.split_group.map(assign).fillna('train')
    signers['fold'] = signers.split_group.map(fold_of)
    split_of, fold_by_signer = dict(zip(signers.signer, signers.split)), dict(zip(signers.signer, signers.fold))

    # task 1 — Words_200 clips (link via tag key → recording → signer)
    w = words.merge(tags[['key', 'rec_id', 'start', 'end']], on='key', how='left')
    w['signer'] = w.rec_id.map(signer_of).fillna('unknown')
    w['split'] = w.signer.map(split_of).fillna('train')
    w['fold'] = w.signer.map(fold_by_signer)
    # task 2 — every annotated gloss cut from the sentences (slice the recording's landmarks)
    n_clips, n_sig = t.groupby('gloss').size(), t.groupby('gloss').signer.nunique()
    t['in_w200'] = t.gloss.isin(vocab)
    t['min_clips'] = t.gloss.map((n_clips >= min_clips) & (n_sig >= 3))
    t['split'], t['fold'] = t.signer.map(split_of), t.signer.map(fold_by_signer)
    t['has_landmarks'] = t.rec_id.isin(m.index)
    # task 3 — whole sentences; a held-out set of sentence folders measures generalisation to unseen sentences
    rng = np.random.default_rng(seed)
    folders = sorted(recs.folder.unique(), key=lambda x: (len(x), x))
    held = set(rng.choice(folders, max(1, int(len(folders) * heldout_sentence_frac)), replace=False))
    s = signers.merge(recs[['rec_id', 'folder', 'frames']], on='rec_id', how='left')
    s['heldout_sentence'] = s.folder.isin(held)
    s = s.merge(sents, on='folder', how='left')      # sentence text + gloss sequence, for Phase 4
    # task 4 — fingerspelling (contributor id or clip run as the group)
    fs = store.LandmarkStore(fs_dirs).manifest
    fs_assign, fs_cover = {}, {}
    for kind, g in fs.groupby('kind'):
        a, c = S.pick_split(g, 'group', 'letter', seed=seed, trials=trials)
        fs_assign.update(a)
        fs_cover[kind] = c
    fs['split'] = fs.group.map(fs_assign)

    os.makedirs(out_dir, exist_ok=True)
    signers.to_csv(os.path.join(out_dir, 'signers.csv'), index=False)
    w.to_csv(os.path.join(out_dir, 'words200_split.csv'), index=False)
    t.to_csv(os.path.join(out_dir, 'glosses_split.csv'), index=False)
    s.to_csv(os.path.join(out_dir, 'sentences_split.csv'), index=False)
    fs[['id', 'letter', 'kind', 'group', 'run', 'frame_no', 'hand_rate', 'split']].to_csv(
        os.path.join(out_dir, 'fingerspelling_split.csv'), index=False)

    split_signers = signers.drop_duplicates('signer').groupby('split').size().to_dict()
    rep['closest_pairs'] = S.closest_pairs(signers, emb_ids, emb)
    cross = [p['sim'] for p in S.closest_pairs(signers, emb_ids, emb, top=1, cross_split_only=True)]
    gsize = signers.groupby('split_group').size()
    rep['split_groups'] = {'count': int(len(gsize)), 'largest_share': round(float(gsize.max() / gsize.sum()), 3),
                           'per_split': signers.drop_duplicates('split_group').groupby('split').size().to_dict(),
                           'max_cross_split_sim': max(cross, default=None), 'link_sim': link_sim}
    bench = {
        'protocol': 'signer-independent; one split-group→split map shared by all tasks; seed %d' % seed,
        'signers': {**rep, 'per_split': split_signers, 'sim_threshold': sim, 'session_gap_min': gap_min},
        'frontal_outliers': int((signers.frontal > 0.35).sum()),
        'words200': {'coverage': cover, 'unlinked_clips': int((w.signer == 'unknown').sum()),
                     **S.baselines(w, 'gloss')},
        'glosses_min%d' % min_clips: S.baselines(t[t.min_clips], 'gloss'),
        'glosses_all': S.baselines(t, 'gloss'),
        'sentences': {'heldout_folders': len(held),
                      **{k: int(v) for k, v in s.groupby('split').size().items()}},
        'fingerspelling': {kind: {'coverage': fs_cover[kind], **S.baselines(g, 'letter')}
                           for kind, g in fs.groupby('kind')},
    }
    with open(os.path.join(out_dir, 'benchmark.json'), 'w', encoding='utf-8') as f:
        json.dump(bench, f, ensure_ascii=False, indent=1, default=str)
    return bench, signers, dict(zip(emb_ids, thumbs))


def signer_sheet(signers, thumbs, per_signer=6, size=72):
    """Contact sheet: one row per pseudo-signer, a few face crops each — to eyeball the clustering.
    Shown in the notebook only; not written to the output."""
    import cv2
    import numpy as np
    rows = []
    for sig, g in signers[signers.rec_id.isin(thumbs)].groupby('signer'):
        tiles = []
        for rid in g.rec_id.sample(min(per_signer, len(g)), random_state=0):
            im = cv2.imdecode(np.frombuffer(thumbs[rid], np.uint8), cv2.IMREAD_COLOR)
            if im is not None:
                tiles.append(cv2.resize(im, (size, size)))
        tiles += [np.zeros((size, size, 3), np.uint8)] * (per_signer - len(tiles))
        label = np.full((size, 90, 3), 255, np.uint8)
        cv2.putText(label, f'{sig} {len(g)}', (2, size // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)
        rows.append(np.hstack([label] + tiles))
    return cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB) if rows else None

In [ ]:
from azsl import pipeline
INPUT = '/kaggle/input' if ON_KAGGLE else OUT

def find(pattern):
    return sorted({os.path.dirname(p) for p in glob.glob(os.path.join(INPUT, '**', pattern), recursive=True)})

index_dir = find('index/tags.csv')
words_dirs = find('landmarks/words200/manifest.csv')
fs_dirs = find('landmarks/fingerspelling/manifest.csv')
sentence_dirs = find('landmarks/sentences_cam2/shard*/manifest.csv')
print('index:', index_dir, '\nwords:', words_dirs, '\nfingerspelling:', fs_dirs, '\nsentence shards:', sentence_dirs)
assert index_dir and words_dirs and fs_dirs and sentence_dirs, 'attach the outputs of notebooks 01 and 02'
if len(sentence_dirs) < 4:
    print(f'WARNING: only {len(sentence_dirs)} of 4 sentence shards attached — signers in the missing ones '
          'fall back to session ids')

In [ ]:
bench, signers, thumbs = pipeline.run_split(index_dir[0], words_dirs, fs_dirs, sentence_dirs,
                                            os.path.join(OUT, 'splits'), sim=SIM, link_sim=LINK_SIM, gap_min=GAP_MIN,
                                            seed=SEED, override_csv=OVERRIDE_CSV)
print(json.dumps({k: v for k, v in bench['signers'].items() if k != 'closest_pairs'}, indent=1, default=str))
import pandas as pd
pd.DataFrame(bench['signers']['closest_pairs'])   # a close pair across splits would be one person leaking

### Check the pseudo-signers
Each row should be **one person**. One person spread over several rows is only harmful if the
rows land in different splits — pseudo-signers with any two sittings at least `LINK_SIM` alike
are tied to one split (`split_group`); `closest_pairs` / `max_cross_split_sim` compare sittings
and show what is left across splits. Two people in one row → raise `SIM`. `nearest_other_signer_sim` above should stay well below `SIM`, and
`signer_sentence_repeats` small (a signer rarely records the same sentence twice).

In [ ]:
import matplotlib.pyplot as plt
sheet = pipeline.signer_sheet(signers, thumbs)
if sheet is not None:
    plt.figure(figsize=(12, max(3, sheet.shape[0] / 60)))
    plt.imshow(sheet); plt.axis('off'); plt.show()
signers.groupby('signer').agg(recordings=('rec_id', 'size'), sessions=('session', 'nunique'),
                              split=('split', 'first')).sort_values('recordings', ascending=False).head(60)

### Benchmark

In [ ]:
for task in ('words200', 'glosses_min20', 'glosses_all'):
    b = bench[task]
    print(f"{task}: majority class {b['majority_class']!r}")
    for s in ('train', 'val', 'test'):
        print(f"   {s:5s} {b[s]}")
print('words200 coverage:', bench['words200']['coverage'], '| unlinked clips (train only):', bench['words200']['unlinked_clips'])
print('sentences:', bench['sentences'])
for kind, b in bench['fingerspelling'].items():
    print(f"fingerspelling {kind}: {b['coverage']} | test {b['test']}")
print('frontal-camera outliers:', bench['frontal_outliers'])